# 12 Employee Skills Baseline Generation
**Enterprise HR AI — Workforce Intelligence & Upskilling Platform**

### Purpose:
Establish a controlled, deterministic employee skills table derived from O*NET role taxonomies and employee tenure, explicitly labeled as synthetic MVP data.


In [2]:
import os
import random
import pandas as pd
import numpy as np

DATA_PROCESSED = "../data/processed"
df_attrition = pd.read_csv(os.path.join(DATA_PROCESSED, "employee_attrition_processed.csv"))
df_profiles = pd.read_csv(os.path.join(DATA_PROCESSED, "role_competency_profiles.csv"))

# Build role -> required skills dictionary
role_skills_map = {}
for _, row in df_profiles.iterrows():
    role_skills_map[row['job_role']] = row['required_skills_list'].split('|')

# Seed for deterministic reproducible baseline
random.seed(42)
np.random.seed(42)

employee_skills_records = []

for _, emp in df_attrition.iterrows():
    emp_id = emp['employee_id']
    role = emp['job_role']
    years_exp = emp['total_working_years']
    
    available_role_skills = role_skills_map.get(role, ['Problem Solving', 'Communication', 'Microsoft Excel'])
    total_skills = len(available_role_skills)
    
    # Experienced employees have acquired more role skills (between 50% and 85%)
    acquisition_rate = min(0.85, max(0.40, 0.40 + (years_exp / 40.0) * 0.45))
    k_skills = max(2, int(total_skills * acquisition_rate))
    
    # Deterministic sample based on employee id seed
    rng = random.Random(emp_id + 42)
    acquired_skills = rng.sample(available_role_skills, min(k_skills, total_skills))
    
    for skill in acquired_skills:
        employee_skills_records.append({
            'employee_id': emp_id,
            'job_role': role,
            'skill_name': skill,
            'proficiency_level': rng.choice(['Intermediate', 'Advanced', 'Proficient']),
            'data_source': 'Controlled_MVP_Benchmark_Seed42'
        })

df_emp_skills = pd.DataFrame(employee_skills_records)
out_emp_skills = os.path.join(DATA_PROCESSED, "employee_skills_controlled.csv")
df_emp_skills.to_csv(out_emp_skills, index=False)

print(f"Generated {len(df_emp_skills):,} skill records across {df_emp_skills['employee_id'].nunique()} employees.")
print(df_emp_skills.head(10).to_string(index=False))


Generated 14,893 skill records across 1470 employees.
 employee_id           job_role           skill_name proficiency_level                     data_source
           1    Sales Executive     Active Listening        Proficient Controlled_MVP_Benchmark_Seed42
           1    Sales Executive    Microsoft Outlook          Advanced Controlled_MVP_Benchmark_Seed42
           1    Sales Executive    Critical Thinking        Proficient Controlled_MVP_Benchmark_Seed42
           1    Sales Executive    Adobe Illustrator      Intermediate Controlled_MVP_Benchmark_Seed42
           1    Sales Executive      Active Learning        Proficient Controlled_MVP_Benchmark_Seed42
           1    Sales Executive Microsoft PowerPoint          Advanced Controlled_MVP_Benchmark_Seed42
           1    Sales Executive  Salesforce software        Proficient Controlled_MVP_Benchmark_Seed42
           1    Sales Executive     Autodesk AutoCAD          Advanced Controlled_MVP_Benchmark_Seed42
           1    Sal